# V3 Training + Evaluation

GATv2 with custom loss: `MSE + 0.2×stillness + 0.05×direction`

- **Stillness**: penalizes large outputs when near goal (`exp(-2d)`)
- **Direction**: cosine alignment with goal (only when `d > 0.5m`)

In [ ]:
import sys, os, json, torch
import torch.nn.functional as F
from pathlib import Path
sys.path.insert(0, '../src')
from torch_geometric.loader import DataLoader
from model import SetpointGATv2
from dataloader import load_splits, DatasetNormalizer, normalize_batch, engineer_x

## Config

In [ ]:
CFG = {
    'in_channels': 64, 'hidden_channels': 64, 'out_channels': 4,
    'edge_dim': 7, 'heads': 4, 'num_layers': 3, 'dropout': 0.1,
    'lr': 1e-3, 'weight_decay': 1e-4, 'batch_size': 64,
    'epochs': 100, 'patience': 15,
    'raw_frame_dim': 35, 'yaw_idx_in_frame': 25,
    'cos_sin_indices': [25, 26, 57, 58], 'yaw_quantile': 0.99,
    'stillness_weight': 0.2, 'direction_weight': 0.05,
    'stillness_decay': 2.0, 'direction_min_dist': 0.5,
}
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## Load Data

In [ ]:
TRAIN = r'../datasets/setpoint_V3_mixed_formations_train.pt'
VAL   = r'../datasets/setpoint_V3_mixed_formations_val.pt'
TEST  = r'../datasets/setpoint_V3_mixed_formations_test.pt'

train_ds, val_ds, test_ds = load_splits(TRAIN, VAL, TEST)
normalizer = DatasetNormalizer.fit(train_ds, CFG).to(device)

train_loader = DataLoader(train_ds, batch_size=CFG['batch_size'], shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=CFG['batch_size'])
test_loader  = DataLoader(test_ds, batch_size=CFG['batch_size'])
print(f'Train: {len(train_ds)}  Val: {len(val_ds)}  Test: {len(test_ds)}')

## Custom Loss

In [ ]:
def compute_loss(pred, target, x_raw, cfg):
    mse = F.mse_loss(pred, target)
    local_pos_err = x_raw[:, 22:25]
    dist = torch.norm(local_pos_err, dim=1)
    # Stillness: penalize large outputs near goal
    stillness = (pred[:, :3].norm(dim=1) * torch.exp(-dist * cfg['stillness_decay'])).mean()
    # Direction: align with goal when far
    far = dist > cfg['direction_min_dist']
    if far.any():
        pn = pred[far, :2].norm(dim=1, keepdim=True).clamp(min=1e-4)
        gn = local_pos_err[far, :2].norm(dim=1, keepdim=True).clamp(min=1e-4)
        direction = 1.0 - F.cosine_similarity(pred[far, :2]/pn, local_pos_err[far, :2]/gn, dim=1).mean()
    else:
        direction = torch.tensor(0.0, device=pred.device)
    total = mse + cfg['stillness_weight'] * stillness + cfg['direction_weight'] * direction
    return total, mse.item(), stillness.item(), direction.item()

## Model + Optimizer

In [ ]:
model = SetpointGATv2(
    in_ch=CFG['in_channels'], hid_ch=CFG['hidden_channels'],
    out_ch=CFG['out_channels'], edge_dim=CFG['edge_dim'],
    heads=CFG['heads'], num_layers=CFG['num_layers'], dropout=CFG['dropout']
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

## Train

In [ ]:
ckpt_dir = Path('../checkpoints'); ckpt_dir.mkdir(exist_ok=True)
best_val, patience_ctr = float('inf'), 0
history = []

for epoch in range(1, CFG['epochs'] + 1):
    model.train()
    t_loss, t_mse, t_still, t_dir, n = 0, 0, 0, 0, 0
    for batch in train_loader:
        batch = batch.to(device)
        x_eng = engineer_x(batch.x, CFG['raw_frame_dim'], CFG['yaw_idx_in_frame'])
        batch = normalize_batch(batch, normalizer)
        pred = model(batch.x, batch.edge_index, batch.edge_attr)
        loss, mv, sv, dv = compute_loss(pred, batch.target, x_eng, CFG)
        optimizer.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        bs = batch.num_graphs
        t_loss += loss.item()*bs; t_mse += mv*bs; t_still += sv*bs; t_dir += dv*bs; n += bs
    t_loss /= n; t_mse /= n; t_still /= n; t_dir /= n

    model.eval()
    v_loss, v_n = 0, 0
    with torch.no_grad():
        for batch in val_loader:
            batch = batch.to(device)
            x_eng = engineer_x(batch.x, CFG['raw_frame_dim'], CFG['yaw_idx_in_frame'])
            batch = normalize_batch(batch, normalizer)
            pred = model(batch.x, batch.edge_index, batch.edge_attr)
            loss, _, _, _ = compute_loss(pred, batch.target, x_eng, CFG)
            bs = batch.num_graphs; v_loss += loss.item()*bs; v_n += bs
    v_loss /= v_n
    scheduler.step(v_loss)
    history.append({'epoch': epoch, 'train': t_loss, 'val': v_loss})

    print(f'Ep {epoch:3d} | Train: {t_loss:.5f} (MSE:{t_mse:.5f} S:{t_still:.5f} D:{t_dir:.5f}) | Val: {v_loss:.5f}')

    if v_loss < best_val:
        best_val = v_loss; patience_ctr = 0
        torch.save(model.state_dict(), ckpt_dir / 'best_gatv2.pth')
        normalizer.save(ckpt_dir / 'normalization_stats.pt')
    else:
        patience_ctr += 1
        if patience_ctr >= CFG['patience']:
            print(f'Early stop at epoch {epoch}'); break

print(f'Best val: {best_val:.5f}')

## Evaluate on Test

In [ ]:
model.load_state_dict(torch.load(ckpt_dir / 'best_gatv2.pth', weights_only=True))
model.eval()
test_loss, test_n = 0, 0
with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(device)
        x_eng = engineer_x(batch.x, CFG['raw_frame_dim'], CFG['yaw_idx_in_frame'])
        batch = normalize_batch(batch, normalizer)
        pred = model(batch.x, batch.edge_index, batch.edge_attr)
        loss, _, _, _ = compute_loss(pred, batch.target, x_eng, CFG)
        bs = batch.num_graphs; test_loss += loss.item()*bs; test_n += bs
print(f'Test loss: {test_loss/test_n:.5f}')

## Loss Curves

In [ ]:
import matplotlib.pyplot as plt
epochs = [h['epoch'] for h in history]
plt.figure(figsize=(8, 4))
plt.plot(epochs, [h['train'] for h in history], label='Train')
plt.plot(epochs, [h['val'] for h in history], label='Val')
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.legend(); plt.title('V3 Training Curves')
plt.tight_layout(); plt.show()

## Save Results

In [ ]:
results_dir = Path('../results'); results_dir.mkdir(exist_ok=True)
with open(results_dir / 'eval_metrics.json', 'w') as f:
    json.dump({'best_val': best_val, 'test_loss': test_loss/test_n, 'config': CFG}, f, indent=2, default=str)
print('Saved results.')